# Jensen-Shannon Divergence: Factuality Score Distributions

Compares full distributions of factuality scores (not just means) between:
- Government vs opposition speakers
- Governing party eras
- Pre/during/post COVID
- Across jurisdictions

JSD = 0 means identical distributions. JSD = 1 means maximally different.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import jensenshannon
from scipy.stats import gaussian_kde

REPO = '/Users/caprice/MiscellaneousProjects/legislative-health-factuality'
os.makedirs(f'{REPO}/figures', exist_ok=True)

print('Libraries loaded')

In [ ]:
# ── Load sentence-level data ──────────────────────────────────────────────────

print('Loading DB1 files...')
with open(f'{REPO}/data/dbpedia_pipeline/ontario/db1_claim_bearing.json') as f:
    ont_db1 = json.load(f)
with open(f'{REPO}/data/dbpedia_pipeline/quebec/db1_claim_bearing.json') as f:
    qc_db1 = json.load(f)
with open(f'{REPO}/data/dbpedia_pipeline/us/db1_claim_bearing.json') as f:
    us_db1 = json.load(f)

print(f'Ontario: {len(ont_db1):,} | Quebec: {len(qc_db1):,} | US: {len(us_db1):,}')

In [ ]:
# ── JSD helper functions ──────────────────────────────────────────────────────

def compute_jsd(scores_a, scores_b, bins=50):
    """Compute JSD between two sets of factuality scores using histogram binning."""
    if len(scores_a) < 10 or len(scores_b) < 10:
        return None
    # Use fixed bins 0-1
    bin_edges = np.linspace(0, 1, bins + 1)
    hist_a, _ = np.histogram(scores_a, bins=bin_edges, density=True)
    hist_b, _ = np.histogram(scores_b, bins=bin_edges, density=True)
    # Add small epsilon to avoid log(0)
    hist_a = hist_a + 1e-10
    hist_b = hist_b + 1e-10
    # Normalize
    hist_a = hist_a / hist_a.sum()
    hist_b = hist_b / hist_b.sum()
    return jensenshannon(hist_a, hist_b)

def get_sentence_scores_by_date(db1):
    """Group sentence-level support values by date."""
    by_date = {}
    for s in db1:
        date = s.get('date', '')
        if not date: continue
        if date not in by_date:
            by_date[date] = []
        by_date[date].append(int(s.get('supported', False)))
    # Compute per-bill factuality score
    return {date: sum(v)/len(v) for date, v in by_date.items() if len(v) >= 5}

def get_sentence_scores_by_party_era(db1, gov_party_fn):
    """Get support values split by government vs opposition."""
    gov_scores = []
    opp_scores = []
    for s in db1:
        date = s.get('date', '')
        party = s.get('party', '')
        if not date or not party: continue
        year = int(date[:4])
        gov_party = gov_party_fn(year)
        if not gov_party: continue
        is_gov = gov_party.lower().split()[0] in party.lower()
        val = int(s.get('supported', False))
        if is_gov:
            gov_scores.append(val)
        else:
            opp_scores.append(val)
    return gov_scores, opp_scores

def get_ontario_gov(year):
    if year < 1986: return 'Progressive Conservative'
    if year < 1990: return 'Liberal'
    if year < 1995: return 'New Democratic'
    if year < 2003: return 'Progressive Conservative'
    if year < 2018: return 'Liberal'
    return 'Progressive Conservative'

def get_quebec_gov(year):
    if year < 1994: return 'Liberal'
    if year < 2003: return 'Parti Quebecois'
    if year < 2012: return 'Liberal'
    if year < 2014: return 'Parti Quebecois'
    if year < 2018: return 'Liberal'
    return 'Coalition Avenir Quebec'

COVID_START = '2020-01-30'
COVID_END   = '2023-05-05'

def get_period(date):
    if date < COVID_START: return 'Pre'
    if date <= COVID_END:  return 'During'
    return 'Post'

print('Helpers defined')

In [ ]:
# ── ANALYSIS 1: JSD between bill-level factuality distributions ───────────────
# Compare across jurisdictions

us_bill_scores  = list(get_sentence_scores_by_date(us_db1).values())
ont_bill_scores = list(get_sentence_scores_by_date(ont_db1).values())
qc_bill_scores  = list(get_sentence_scores_by_date(qc_db1).values())

print('=== JSD: ACROSS JURISDICTIONS ===')
print(f'US vs Ontario:  JSD = {compute_jsd(us_bill_scores, ont_bill_scores):.4f}')
print(f'US vs Quebec:   JSD = {compute_jsd(us_bill_scores, qc_bill_scores):.4f}')
print(f'Ontario vs Quebec: JSD = {compute_jsd(ont_bill_scores, qc_bill_scores):.4f}')
print()
print(f'N: US={len(us_bill_scores)}, Ontario={len(ont_bill_scores)}, Quebec={len(qc_bill_scores)}')

In [ ]:
# ── ANALYSIS 2: JSD government vs opposition ──────────────────────────────────

print('=== JSD: GOVERNMENT VS OPPOSITION ===')

ont_gov, ont_opp = get_sentence_scores_by_party_era(ont_db1, get_ontario_gov)
qc_gov, qc_opp   = get_sentence_scores_by_party_era(qc_db1, get_quebec_gov)

print(f'Ontario Gov vs Opp: JSD = {compute_jsd(ont_gov, ont_opp):.4f}')
print(f'  Gov n={len(ont_gov):,}, Opp n={len(ont_opp):,}')
print(f'Quebec Gov vs Opp:  JSD = {compute_jsd(qc_gov, qc_opp):.4f}')
print(f'  Gov n={len(qc_gov):,}, Opp n={len(qc_opp):,}')

# By era for Ontario
print()
print('Ontario JSD by era:')
ont_eras = [
    ('Liberal 1987-1990', 1987, 1990),
    ('NDP 1990-1995',     1990, 1995),
    ('PC 1995-2003',      1995, 2003),
    ('Liberal 2003-2018', 2003, 2018),
    ('PC 2018+',          2018, 2030),
]
for era_name, start, end in ont_eras:
    era_gov = [int(s.get('supported',False)) for s in ont_db1
               if start <= int(s.get('date','0')[:4]) < end
               and get_ontario_gov(int(s.get('date','0')[:4])).lower().split()[0] 
               in s.get('party','').lower()]
    era_opp = [int(s.get('supported',False)) for s in ont_db1
               if start <= int(s.get('date','0')[:4]) < end
               and get_ontario_gov(int(s.get('date','0')[:4])).lower().split()[0] 
               not in s.get('party','').lower()]
    jsd = compute_jsd(era_gov, era_opp)
    if jsd is not None:
        print(f'  {era_name}: JSD={jsd:.4f} (gov n={len(era_gov):,}, opp n={len(era_opp):,})')

In [ ]:
# ── ANALYSIS 3: JSD across COVID periods ─────────────────────────────────────

print('=== JSD: COVID PERIODS ===')
for name, db1 in [('Ontario', ont_db1), ('Quebec', qc_db1), ('US', us_db1)]:
    pre    = [int(s.get('supported',False)) for s in db1 if s.get('date','') < COVID_START]
    during = [int(s.get('supported',False)) for s in db1 
              if COVID_START <= s.get('date','') <= COVID_END]
    post   = [int(s.get('supported',False)) for s in db1 if s.get('date','') > COVID_END]
    
    print(f'{name}:')
    print(f'  Pre vs During:  JSD = {compute_jsd(pre, during):.4f} (n={len(pre):,} vs {len(during):,})')
    print(f'  Pre vs Post:    JSD = {compute_jsd(pre, post):.4f} (n={len(pre):,} vs {len(post):,})')
    print(f'  During vs Post: JSD = {compute_jsd(during, post):.4f} (n={len(during):,} vs {len(post):,})')
    print()

In [ ]:
# ── FIGURE 1: Distribution plots ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribution of Bill-Level Factuality Scores by Jurisdiction', 
             fontsize=13, fontweight='bold')

for ax, (name, scores, color) in zip(axes, [
    ('US House', us_bill_scores, '#2166AC'),
    ('Ontario LA', ont_bill_scores, '#B2182B'),
    ('Quebec NA', qc_bill_scores, '#1A9641'),
]):
    ax.hist(scores, bins=30, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.axvline(np.mean(scores), color='black', linewidth=2, linestyle='--',
               label=f'Mean: {np.mean(scores):.3f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Factuality Score')
    ax.set_ylabel('Number of Bills')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPO}/figures/factuality_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/factuality_distributions.png')

In [ ]:
# Bill-level gov vs opp factuality distributions
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Government vs Opposition Bill-Level Factuality Distributions',
             fontsize=12, fontweight='bold')

for ax, (name, db1, gov_fn) in zip(axes, [
    ('Ontario LA', ont_db1, get_ontario_gov),
    ('Quebec NA',  qc_db1,  get_quebec_gov),
]):
    from collections import defaultdict
    gov_by_date = defaultdict(lambda: [0,0])
    opp_by_date = defaultdict(lambda: [0,0])
    
    for s in db1:
        date = s.get('date','')
        party = s.get('party','').lower()
        if not date or not party: continue
        year = int(date[:4])
        g = gov_fn(year).lower().split()[0]
        val = int(s.get('supported', False))
        if g in party:
            gov_by_date[date][0] += val
            gov_by_date[date][1] += 1
        else:
            opp_by_date[date][0] += val
            opp_by_date[date][1] += 1
    
    gov_scores = np.array([v[0]/v[1] for v in gov_by_date.values() if v[1] >= 5])
    opp_scores = np.array([v[0]/v[1] for v in opp_by_date.values() if v[1] >= 5])
    jsd_val = compute_jsd(gov_scores.tolist(), opp_scores.tolist())
    
    x = np.linspace(0, 0.5, 200)
    
    kde_gov = gaussian_kde(gov_scores, bw_method=0.15)
    kde_opp = gaussian_kde(opp_scores, bw_method=0.15)
    
    ax.plot(x, kde_gov(x), color='#1A1A1A', linewidth=2.5,
            label=f'Government (n={len(gov_scores)}, mean={gov_scores.mean():.3f})')
    ax.plot(x, kde_opp(x), color='#888888', linewidth=2.5, linestyle='--',
            label=f'Opposition (n={len(opp_scores)}, mean={opp_scores.mean():.3f})')
    ax.fill_between(x, kde_gov(x), kde_opp(x), alpha=0.15, color='blue',
                    label=f'Difference (JSD={jsd_val:.4f})')
    
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Bill-Date Factuality Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 0.5)

plt.tight_layout()
plt.savefig(f'{REPO}/figures/gov_vs_opp_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [2]:
print('Quebec JSD by era:')
qc_eras = [
    ('PQ 1994-2003',      1994, 2003),
    ('Liberal 2003-2012', 2003, 2012),
    ('PQ 2012-2014',      2012, 2014),
    ('Liberal 2014-2018', 2014, 2018),
    ('CAQ 2018+',         2018, 2030),
]
for era_name, start, end in qc_eras:
    qg = [int(s.get('supported',False)) for s in qc_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_quebec_gov(int(s.get('date','0')[:4])).lower().split()[0]
          in s.get('party','').lower()]
    qo = [int(s.get('supported',False)) for s in qc_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_quebec_gov(int(s.get('date','0')[:4])).lower().split()[0]
          not in s.get('party','').lower()]
    jsd = compute_jsd(qg, qo)
    if jsd is not None:
        print(f'  {era_name}: JSD={jsd:.4f} (gov n={len(qg):,}, opp n={len(qo):,})')

Quebec JSD by era:


NameError: name 'qc_db1' is not defined

In [ ]:
# ── FIGURE 3: JSD over time (rolling) ────────────────────────────────────────
# Compare gov vs opp JSD by decade to show if divergence is changing

print('=== JSD OVER TIME: GOVERNMENT VS OPPOSITION ===')

decades = [(1985,1995), (1995,2005), (2005,2015), (2015,2026)]
decade_labels = ['1985-1994', '1995-2004', '2005-2014', '2015-2026']

ont_jsd_by_decade = []
qc_jsd_by_decade = []

for (start, end), label in zip(decades, decade_labels):
    # Ontario
    og = [int(s.get('supported',False)) for s in ont_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_ontario_gov(int(s.get('date','0')[:4])).lower().split()[0]
          in s.get('party','').lower()]
    oo = [int(s.get('supported',False)) for s in ont_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_ontario_gov(int(s.get('date','0')[:4])).lower().split()[0]
          not in s.get('party','').lower()]
    jsd_ont = compute_jsd(og, oo)
    ont_jsd_by_decade.append(jsd_ont)
    
    # Quebec  
    qg = [int(s.get('supported',False)) for s in qc_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_quebec_gov(int(s.get('date','0')[:4])).lower().split()[0]
          in s.get('party','').lower()]
    qo = [int(s.get('supported',False)) for s in qc_db1
          if start <= int(s.get('date','0')[:4]) < end
          and get_quebec_gov(int(s.get('date','0')[:4])).lower().split()[0]
          not in s.get('party','').lower()]
    jsd_qc = compute_jsd(qg, qo)
    qc_jsd_by_decade.append(jsd_qc)
    
    print(f'{label}: Ontario JSD={jsd_ont:.4f if jsd_ont else "n/a"}, Quebec JSD={jsd_qc:.4f if jsd_qc else "n/a"}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(decade_labels))
ax.plot(x, [j if j else 0 for j in ont_jsd_by_decade], 'o-', color='#B2182B',
        linewidth=2.5, markersize=8, label='Ontario LA')
ax.plot(x, [j if j else 0 for j in qc_jsd_by_decade], 's-', color='#1A9641',
        linewidth=2.5, markersize=8, label='Quebec NA')
ax.set_xticks(x)
ax.set_xticklabels(decade_labels)
ax.set_ylabel('Jensen-Shannon Divergence')
ax.set_title('Government vs Opposition Factuality Distribution Divergence Over Time\n'
             'Higher JSD = more different distributions', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig(f'{REPO}/figures/jsd_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/jsd_over_time.png')

In [ ]:
# ── US placeholder ────────────────────────────────────────────────────────────
# US gov vs opp JSD requires speaker-level attribution
# Speaker turns available for 91/145 health bill dates at 84% party match
# Full analysis pending complete speaker attribution

print('US JSD: pending complete speaker attribution')
print('Current coverage: 91/145 health bill dates, 84% party match')
print('Will add once speaker attribution is complete')